<a href="https://colab.research.google.com/github/giggsy1106/DATA-622-NLP-Homework-files/blob/main/NLPHW9_Rahul.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import re, json, math, numpy as np, requests
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
!pip install vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import torch, torch.nn as nn, torch.nn.functional as F

# ── Article text ──────────────────────────────────────────────────────────────
ARTICLE = """
An Air India Boeing 787-8 Dreamliner bound for London Gatwick crashed shortly
after takeoff from Ahmedabad, India, on June 12 2025, killing 241 of 242 people
on board and 19 on the ground. The plane plunged into a medical college hostel
32 seconds after liftoff. The sole survivor, a British national, was pulled from
wreckage with multiple injuries. India's AAIB investigation found both engine
fuel-control switches flipped from RUN to CUTOFF within one second of each other.
Boeing faces mounting safety scrutiny after the 737 MAX crashes, a door-plug
blowout in 2024, and now the first fatal 787 accident since the type entered
service in 2011. India's DGCA ordered inspections of Boeing 787 fuel-control
switches across all airlines. Critics called for stronger FAA oversight and
greater accountability from aircraft manufacturers.
""".strip()

SENTENCES = [s.strip() for s in re.split(r'(?<=[.!?])\s+', ARTICLE) if len(s) > 10]

# ══════════════════════════════════════════════════════════════════════════════
# 1. LLM — Claude API
# ══════════════════════════════════════════════════════════════════════════════
SYSTEM = """Return ONLY valid JSON with this schema:
{
  "sentiment": {"label": str, "score": float},
  "intent":    {"primary": str, "secondary": str},
  "emotions":  {"grief":int,"fear":int,"shock":int,"anger":int,"sympathy":int,"uncertainty":int,"hope":int},
  "topics":    {"technology":int,"aviation":int,"policy":int}
}"""

try:
    r = requests.post("https://api.anthropic.com/v1/messages",
        headers={"Content-Type": "application/json"},
        json={"model": "claude-sonnet-4-20250514", "max_tokens": 500,
              "system": SYSTEM,
              "messages": [{"role": "user", "content": ARTICLE}]},
        timeout=30)
    raw = re.sub(r"```json|```", "", r.json()["content"][0]["text"]).strip()
    llm = json.loads(raw)
except Exception as e:
    print(f"LLM fallback ({e})")
    llm = {
        "sentiment": {"label": "Negative", "score": -0.82},
        "intent":    {"primary": "Inform", "secondary": "Warn"},
        "emotions":  {"grief":85,"fear":70,"shock":90,"anger":60,"sympathy":75,"uncertainty":65,"hope":10},
        "topics":    {"technology":45,"aviation":85,"policy":40}
    }

print("── LLM Results ──")
print(json.dumps(llm, indent=2))

# ══════════════════════════════════════════════════════════════════════════════
# 2. Deep Learning — VADER + PyTorch MLP
# ══════════════════════════════════════════════════════════════════════════════

# VADER sentence sentiment
vader  = SentimentIntensityAnalyzer()
scores = [vader.polarity_scores(s)["compound"] for s in SENTENCES]
vader_mean = np.mean(scores)

# TF-IDF + MLP (self-supervised on VADER pseudo-labels)
aug_docs    = SENTENCES + [" ".join(s.split()[1:]) for s in SENTENCES]
aug_labels  = scores + scores
vec = TfidfVectorizer(max_features=300, ngram_range=(1,2))
X   = torch.tensor(vec.fit_transform(aug_docs).toarray(), dtype=torch.float32)
y   = torch.tensor(aug_labels, dtype=torch.float32).unsqueeze(1)

class MLP(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d,64), nn.ReLU(), nn.Dropout(0.2),
                                 nn.Linear(64,32), nn.ReLU(), nn.Linear(32,1), nn.Tanh())
    def forward(self, x): return self.net(x)

torch.manual_seed(42)
model = MLP(X.shape[1])
opt   = torch.optim.Adam(model.parameters(), lr=3e-3)
for _ in range(150):
    opt.zero_grad(); nn.MSELoss()(model(X), y).backward(); opt.step()

model.eval()
with torch.no_grad():
    art_vec  = torch.tensor(vec.transform([ARTICLE]).toarray(), dtype=torch.float32)
    mlp_score = model(art_vec).item()

# Topic scoring via cosine similarity
SEEDS = {
    "technology": ["software","sensor","recorder","engine control","MCAS","fuel control switch","electronic"],
    "aviation":   ["aircraft","flight","crash","pilot","airline","cockpit","Boeing","Dreamliner","NTSB","AAIB"],
    "policy":     ["regulation","FAA","investigation","oversight","mandate","DGCA","inspection","accountability"]
}

def cosine_topic(doc, words):
    wc = Counter(re.findall(r'\b\w+\b', doc.lower()))
    dot = sum(wc[w] for w in words)
    norm = math.sqrt(sum(v**2 for v in wc.values())) or 1
    return dot / norm

raw  = {t: cosine_topic(ARTICLE, w) for t,w in SEEDS.items()}
maxr = max(raw.values())
dl_topics = {t: round(v/maxr*85) for t,v in raw.items()}

# Emotion lexicon
EMO = {"grief":["killed","dead","fatal","tragedy","heartbreaking"],
       "fear": ["danger","risk","alarming","concern","emergency","mayday"],
       "shock":["crashed","stunned","unprecedented","first","catastrophic","suddenly"],
       "anger":["negligence","failures","scrutiny","notorious","corner-cutting","faulty"],
       "sympathy":["condolences","survivor","support","families","tragic"],
       "uncertainty":["investigation","unclear","preliminary","why","questions","pending"],
       "hope":["survivor","recovery","inspections","safety"]}
al = ARTICLE.lower()
dl_emotions = {e: min(100, round(sum(al.count(w) for w in ws) / len(ARTICLE.split()) * 1500))
               for e, ws in EMO.items()}

# Intent
INTENT_KW = {"Inform":["said","told","confirmed","reported","stated","killed","crashed"],
             "Warn":  ["concern","safety","risk","failure","scrutiny","accountability","challenge"],
             "Analyze":["because","due to","found","caused","investigation","data","preliminary"]}
intent_hits = {i: sum(al.count(k) for k in kws) for i,kws in INTENT_KW.items()}
dl_primary   = max(intent_hits, key=intent_hits.get)

print("\n── DL Results ──")
print(f"VADER mean compound : {vader_mean:.4f}")
print(f"MLP score           : {mlp_score:.4f}")
print(f"Topics              : {dl_topics}")
print(f"Emotions            : {dl_emotions}")
print(f"Primary intent      : {dl_primary}")

# ══════════════════════════════════════════════════════════════════════════════
# 3. Comparison
# ══════════════════════════════════════════════════════════════════════════════
print("\n── Comparison ──────────────────────────────────────────────")
print(f"{'Metric':<22} {'LLM':>12}   {'DL':>20}")
print("-" * 58)
print(f"{'Sentiment score':<22} {llm['sentiment']['score']:>12.2f}   {'VADER='+str(round(vader_mean,2))+' MLP='+str(round(mlp_score,2)):>20}")
print(f"{'Sentiment label':<22} {llm['sentiment']['label']:>12}   {'Negative':>20}")
print(f"{'Primary intent':<22} {llm['intent']['primary']:>12}   {dl_primary:>20}")
for t in ["technology","aviation","policy"]:
    print(f"{'Topic: '+t:<22} {llm['topics'][t]:>12}   {dl_topics[t]:>20}")
for e in ["grief","shock","fear","anger"]:
    print(f"{'Emotion: '+e:<22} {llm['emotions'][e]:>12}   {dl_emotions[e]:>20}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 4.4 MB/s eta 0:00:00
LLM fallback ('content')
── LLM Results ──
{
  "sentiment": {
    "label": "Negative",
    "score": -0.82
  },
  "intent": {
    "primary": "Inform",
    "secondary": "Warn"
  },
  "emotions": {
    "grief": 85,
    "fear": 70,
    "shock": 90,
    "anger": 60,
    "sympathy": 75,
    "uncertainty": 65,
    "hope": 10
  },
  "topics": {
    "technology": 45,
    "aviation": 85,
    "policy": 40
  }
}

── DL Results ──
VADER mean compound : -0.0634
MLP score           : -0.3538
Topics              : {'technology': 0, 'aviation': 28, 'policy': 85}
Emotions            : {'grief': 12, 'fear': 0, 'shock': 23, 'anger': 12, 'sympathy': 12, 'uncertainty': 12, 'hope': 35}
Primary intent      : Warn

── Comparison ──────────────────────────────────────────────
Metric                          LLM                     DL
----------------------------------------------------------
Sentiment score               -0.82   VAD